In [3]:
from pyspark.sql import SparkSession
from pathlib import Path

# Initialize a Spark session
spark = SparkSession.builder.appName("FragranceDataCleaning").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Define the file path
relative_path = "Documents/GitHub/sniffers/data/frag_raw.csv"
file_path = str(Path.home() / relative_path)

try:
    # Read the CSV file into a DataFrame
    frag_raw_df = spark.read.option("header", "true").csv(file_path)
    frag_raw_df.show(10, truncate=False)

    print("\n ======================================= \n Data Types:")
    print(frag_raw_df.dtypes)
finally:
    spark.stop()

+----------------------------------------+-----------------+------+------------+---------------------------------------------------------------------------------------------------------------+---------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------+
|name                                    |gender           |rating|rating_count|main_accords                                                                                                   |perfumers|description                                                                                                                                                                                                                                          

In [9]:
from pyspark.sql import SparkSession
from pathlib import Path

# Initialize a Spark session
spark = SparkSession.builder.appName("FragranceDataCleaning").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Define the file path
relative_path = "Documents/GitHub/sniffers/data/frag_raw.csv"
file_path = str(Path.home() / relative_path)

try:
    frag_raw_df = spark.read.csv(file_path, header=True, inferSchema=True)
    frag_raw_df.createOrReplaceTempView("frag_raw")

    cleaned_df = spark.sql("""
        SELECT
            REGEXP_EXTRACT(url, '([a-zA-Z0-9]+)\\.html$', 1) AS id,         
            SUBSTRING_INDEX(SUBSTRING_INDEX(description, 'by', 1), 'is a', 1) AS name,
            SUBSTRING_INDEX(SUBSTRING_INDEX(description, 'by', -1), 'is a', 1) AS brand,
            REGEXP_EXTRACT(description, 'was launched in ([0-9]{4})', 1) AS release_year,
            CASE
                WHEN name LIKE '%for women and men' THEN 'unisex'
                WHEN name LIKE '%for women' THEN 'women'
                WHEN name LIKE '%for men' THEN 'men'
                ELSE NULL
            END AS gender,
            TRY_CAST(REGEXP_REPLACE(rating_count, ',', '') AS INT) AS rating_count,
            REPLACE(REPLACE(REPLACE(main_accords, '[', ''), ']', ''),"'", "") AS main_accords, -- remove sq brackets
            lower(replace(regexp_extract(description, '(?i)top note[s]? (is|are) ([^.;]*)', 2), ' and', ',')) as top_notes,
            lower(replace(regexp_extract(description, '(?i)middle note[s]? (is|are) ([^.;]*)', 2), ' and', ',')) as mid_notes,
            lower(replace(regexp_extract(description, '(?i)base note[s]? (is|are) ([^.;]*)', 2), ' and', ',')) as base_notes,
            description,
            url
        FROM frag_raw
    """)

    cleaned_df.show(truncate=False)

    print("Schema:")
    cleaned_df.printSchema()

    print("\n ======================================= \n Data Types:")
    print(cleaned_df.dtypes)


except Exception as e:
    print(f"An error occurred: {e}")


+-----+-----------------+--------------------+------------+------+------------+-----------------------------------------------------------------------------------------+------------------------------------------------------------------------+-------------------------------------------------------------------------------------------+----------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------+
|id   |name             |brand               |release_year|gender|rating_count|main_accords                                                                      

In [ ]:

from pyspark.sql.functions import split, trim, to_json, col, regexp_replace

def to_json_array(df, col_name, new_col_name):
    return df.withColumn(
        new_col_name,
        to_json(
            split(
                trim(regexp_replace(col(col_name), r"[\[\]']", "")),
                r",\s*"
            )
        )
    )

# Apply to all relevant columns
cleaned_df = to_json_array(cleaned_df, "main_accords", "main_accords_json")
cleaned_df = to_json_array(cleaned_df, "top_notes", "top_notes_json")
cleaned_df = to_json_array(cleaned_df, "mid_notes", "mid_notes_json")
cleaned_df = to_json_array(cleaned_df, "base_notes", "base_notes_json")

cleaned_df.show(n=10, truncate=False)


# cleaned_df.toPandas().to_csv("frag_cleaned_sh.csv", index=False)
# print("\n======================================= \n\nCleaned data written to frag_cleaned_sh.csv")

# print("\n======================================= \n\nData Types:")
# print(cleaned_df.dtypes)


+-----+-----------------+-----+------------+------+------------+-----------------------------------------------------------------------------------------+----------------------------------------------+-------------------------------------------+--------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------+-----------------------------------------------------+--------------------------------------------------+---------------------------------------------+
|id   |name             |brand|release_year|gender|rating_count|main_accords                                                                             |top_notes                   



Cleaned data written to frag_cleaned_sh.csv


Data Types:
[('id', 'string'), ('name', 'string'), ('brand', 'string'), ('release_year', 'string'), ('gender', 'string'), ('rating_count', 'int'), ('main_accords', 'string'), ('top_notes', 'string'), ('mid_notes', 'string'), ('base_notes', 'string'), ('url', 'string'), ('main_accords_json', 'string'), ('top_notes_json', 'string'), ('mid_notes_json', 'string'), ('base_notes_json', 'string')]


In [6]:
cleaned_df.createOrReplaceTempView("temp_view")
filtered_df = spark.sql("""
        SELECT
            *
        FROM temp_view
        WHERE `main_accords_json` LIKE '%this perfume is%'
                        or `top_notes_json` LIKE '%this perfume is%'
                        or `mid_notes_json` LIKE '%this perfume is%'
                        or `base_notes_json` LIKE '%this perfume is%'
    """)

filtered_df.show(truncate=False)
print(f"Number of rows in filtered_df: {filtered_df.count()}")

+----+------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+------+------------+-----------------------------------------------------------------------------------------------+-------------------------------------------------------+-----------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------

Number of rows in filtered_df: 58


In [7]:
# Get all unique accords and notes from the JSON columns
from pyspark.sql.functions import explode, from_json, array_distinct
from pyspark.sql.types import ArrayType, StringType

# Parse JSON arrays back to Spark arrays and explode
def get_unique_from_json_col(df, json_col):
    return (
        df
        .withColumn("arr", from_json(col(json_col), ArrayType(StringType())))
        .select(explode(col("arr")).alias("item"))
        .distinct()
        .select("item")
    )

# Collect unique values from each column
main_accords_unique = get_unique_from_json_col(cleaned_df, "main_accords_json")
top_notes_unique = get_unique_from_json_col(cleaned_df, "top_notes_json")
mid_notes_unique = get_unique_from_json_col(cleaned_df, "mid_notes_json")
base_notes_unique = get_unique_from_json_col(cleaned_df, "base_notes_json")

# Union all and get unique values
all_unique = (
    main_accords_unique
    .union(top_notes_unique)
    .union(mid_notes_unique)
    .union(base_notes_unique)
    .distinct()
    .orderBy("item")
)

all_unique_list = [row.item for row in all_unique.collect()]
print("\n======================================= \n \nAll Unique Accords and Notes:")
print(all_unique_list)
print(f"Number of distinct items: {len(all_unique_list)}")


 
All Unique Accords and Notes:
['', 'Champagne', 'Pear', 'absinthe', 'acai berry', 'accord eudora®', 'acerola', 'acerola blossom', 'acetylfuran', 'acácia', 'african freesia petals', 'african geranium', 'african ginger', 'african orange flower', 'african violet', 'agarwood', 'agarwood (oud)', 'agave', 'agave nectar', 'aglaia', 'akigalawood', 'albizia', 'alcohol', 'aldehydes', 'aldehydic', 'aldron', 'algae', 'algerian geranium', 'allspice', 'almiscarado', 'almond', 'almond blossom', 'almond cream', 'almond milk', 'almond tree', 'almond wood', 'aloe vera', 'alpinia', 'althaea', 'aluminum', 'alumroot', 'alyssum', 'amalfi lemon', 'amaranth', 'amaretto', 'amaryllis', 'amazon lily', 'ambarado', 'amber', 'amber from tunis', 'amber oil', 'amber this perfume is the winner of awardfifi award fragrance of the year men`s nouveau niche 2005', 'amber xtreme', 'ambergris', 'ambertonic', 'amberwood', 'ambrarome', 'ambreine', 'ambretone', 'ambrette', 'ambrette (musk mallow)', 'ambrette (musk mallow) t

In [8]:
import pandas as pd
import json

# Write the unique accords and notes to a JSON file
with open("all_unique_accords_and_notes.json", "w", encoding="utf-8") as f:
    json.dump(all_unique_list, f, ensure_ascii=False, indent=2)

print("Unique accords and notes written to all_unique_accords_and_notes.json")


Unique accords and notes written to all_unique_accords_and_notes.json


In [48]:
spark.stop()
print("Spark session stopped.")

Spark session stopped.
